In [12]:
import sqlite3
import pandas as pd
connection = sqlite3.connect('sales.db')
query = "SELECT * FROM prediction"
df = pd.read_sql_query(query, connection)
print(df.head())
connection.close()

      Order_ID    Product_ID       User_ID  Order_Date Return_Date  \
0  ORD00000000  PROD00000000  USER00000000  2023-08-05  2024-08-26   
1  ORD00000001  PROD00000001  USER00000001  2023-10-09  2023-11-09   
2  ORD00000002  PROD00000002  USER00000002  2023-05-06        None   
3  ORD00000003  PROD00000003  USER00000003  2024-08-29        None   
4  ORD00000004  PROD00000004  USER00000004  2023-01-16        None   

  Product_Category  Product_Price  Order_Quantity Return_Reason Return_Status  \
0         Clothing         411.59               3  Changed mind      Returned   
1            Books         288.88               3    Wrong item      Returned   
2             Toys         390.03               5          None  Not Returned   
3             Toys         401.09               3          None  Not Returned   
4            Books         110.09               4          None  Not Returned   

   Days_to_Return  User_Age User_Gender User_Location Payment_Method  \
0           387.0   

In [13]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [ ]:
columns_to_drop = ['Order_ID', 'Product_ID', 'User_ID', 'Order_Date', 'Return_Date', 'Return_Reason', 'Days_to_Return']
df_clean = df.drop(columns=columns_to_drop)

# 2. Define our target and features
target_col = 'Return_Status'

# Convert target to binary (0 = Not Returned, 1 = Returned)
df_clean[target_col] = df_clean[target_col].map({'Not Returned': 0, 'Returned': 1})

# Separate Continuous and Categorical columns
categorical_cols = ['Product_Category', 'User_Gender', 'User_Location', 'Payment_Method', 'Shipping_Method']
continuous_cols = ['Product_Price', 'Order_Quantity', 'User_Age', 'Discount_Applied']

# 3. One-hot encode categorical features and scale continuous features
df_processed = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

scaler = StandardScaler()
df_processed[continuous_cols] = scaler.fit_transform(df_processed[continuous_cols])

# 4. Split into X and y
X = df_processed.drop(columns=[target_col]).values
y = df_processed[target_col].values

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
class ECommerceDataset(Dataset):
    def __init__(self, X, y):
        # Convert data to PyTorch tensors
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1) # Reshape for BCE loss

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create DataLoaders
train_dataset = ECommerceDataset(X_train, y_train)
test_dataset = ECommerceDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
class TabularClassifier(nn.Module):
    def __init__(self, input_dim):
        super(TabularClassifier, self).__init__()
        
        # Define the network architecture
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),
            
            nn.Linear(64, 32),
            nn.ReLU(),
            
            # Final output layer (1 neuron for binary classification)
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

# Initialize the model
input_features = X_train.shape[1]
model = TabularClassifier(input_dim=input_features)
print(model)

In [ ]:
# Setup loss function and optimizer
criterion = nn.BCELoss() # Combines Sigmoid and Binary Cross Entropy
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
epochs = 100
for epoch in range(epochs):
    model.train() # Set model to training mode
    total_loss = 0
    
    for batch_X, batch_y in train_loader:
        # 1. Forward pass
        predictions = model(batch_X)
        
        # 2. Calculate Loss
        loss = criterion(predictions, batch_y)
        
        # 3. Backward pass and optimization
        optimizer.zero_grad() # Clear old gradients
        loss.backward()       # Calculate new gradients
        optimizer.step()      # Update weights
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}')

print("Training Complete!")